# README

Consumer Lending Regulatory RAG — Colab Submission Guide

This notebook documents the project. It does not call an API. Run the project in `02_Main_Code_and_API_V3.1.ipynb`; read the full source and module explanations in `06_Module_Source_and_Explanation_V3.1.ipynb`.

## Project version and completion status

The adopted pipeline is **V3.1: V3 with the missing Q36 answer repaired**. V5 is retained as an answer-generation comparison. Reorganizing the submission into notebooks does not create a new experiment or change the recorded results.

The offline workflow has been executed and checked locally. The API entry point is included, but a new paid experiment has not been run for this package. Human verification of gold labels and answer reviews, confirmation of the report, and the actual demonstration video remain outstanding. See `00_Status_and_Submission_Map_V3.1.ipynb` and `07_Human_Checks_and_Final_Checklist_V3.1.ipynb` for details.

## Running in Colab

1. Download and extract the submission ZIP. In Colab, select **File → Upload notebook**.
2. Upload `02_Main_Code_and_API_V3.1.ipynb`. The original data and cached experiment are embedded; no separate ZIP or JSON upload is required.
3. Run the first three code cells in order to load the project, reproduce retrieval rankings for all 50 questions, and inspect the Q12 failure and Q01 answer with source evidence.
4. For the cached demonstration, leave the last cell at `RUN_LIVE = False`.
5. For a new paid API run, set `RUN_LIVE = True` and choose `N_QUESTIONS`. Run the cell and enter your OpenRouter API key in the **OpenRouter key (hidden)** prompt. Keep the key out of notebook source and shared files.
6. After the run, inspect the completion status, errors, and retrieval metrics. Download the new results ZIP. New outputs must be evaluated separately and do not inherit historical scores.

The default API run processes 5 questions. Set `N_QUESTIONS = 50` for the complete test. A five-question demonstration is not a replacement for the 50-question evaluation.

## Dependencies and runtime paths

Offline reproduction uses NumPy and the Python standard library. API execution also requires `requests`. The main notebook checks NumPy before setup and checks `requests` before a live run. If a dependency is missing, follow its installation message (`%pip install numpy` or `%pip install requests`). Documentation tables use IPython display; pandas is not required.

In Colab, the embedded project is extracted to `/content/consumer_lending_demo`. Each API run uses a new directory with a timestamp and random suffix, preserving the original experiment. The regulatory evaluation date remains **2026-10-02**; a new run does not automatically update the corpus.

## Intended user, inputs, and outputs

**User:** a consumer-lending compliance analyst or business practitioner.

**Inputs:** a Chinese business question, the fixed regulatory corpus, and the evaluation date.

**Outputs:** ranked regulatory provisions, original source evidence, and an AI explanation with provision-ID citations. A person makes the final regulatory judgment. The system does not approve loans or execute banking transactions.

Regulatory source text and AI interpretation are presented separately. Instructions appearing inside source documents are treated as document content. Conclusions unsupported by evidence should retain explicit uncertainty.

## Architecture

```text
8 regulatory documents / 199 provision chunks
                    |
       Shared retrieval text and metadata
                    |
Business question --+--> Embeddings + cosine ranking --+
                    +--> Chinese BM25 keyword ranking -+
                                                       |
                        Candidate fusion + adjacent provisions
                                                       |
                         LLM evidence selection (up to 10)
                                                       |
                           Evidence-based draft + review
                                                       |
                         Source text + AI interpretation
                                                       |
                                              Human judgment

Separate gold labels --> Top-5 evaluation
Gold labels are not passed to the answer generator.
```

## Corpus and evaluation

The submission contains 8 regulatory documents, 199 provision chunks, and 50 Chinese hypothetical questions. Document names, metadata, URLs, original provisions, and questions are available in `03_Data_and_Corpus_V3.1.ipynb`. Original gold labels and metric definitions are in `04_Evals_and_Gold_V3.1.ipynb`.

Both semantic and keyword retrieval use the same retrieval text fields. The keyword baseline is BM25 using Chinese character bigrams and alphanumeric tokens; it does not use an LLM. The hybrid pipeline combines semantic and lexical candidates, supplements adjacent provisions, and uses an LLM to select evidence.

All 50 questions were used during development and tuning. These results are development-set measurements, not evidence of performance on an independent held-out test set. Gold labels require human validation; D07's joint issuers are now confirmed in notebook 08.

## Models, metrics, and costs

- Embeddings: `openai/text-embedding-3-small`.
- Evidence selection, answer generation, and review: `openai/gpt-4.1-mini`.

These models were used in the recorded experiment. A comprehensive comparison across models has not been performed.

| Retrieval method | Gold hit in top five | Hit@5 |
|---|---:|---:|
| Keyword baseline | 39/50 | 78% |
| Semantic retrieval | 41/50 | 82% |
| Hybrid candidates with LLM evidence selection | 49/50 | 98% |

Coverage within the final evidence set of up to 10 provisions is 50/50 (100%). That is a different metric from Hit@5. Retrieval success does not establish answer correctness. Answer reviews are supplementary AI assessments; human review has not yet been completed. Alternative scoring dimensions are disclosed as post-hoc analysis.

The original V3 cache records USD 0.40287682 in provider-reported usage costs. This is a cache-scoped figure, not the complete project bill. Detailed cost scopes and comparisons are presented in `05_Results_and_Failure_Analysis_V3.1.ipynb`. Hybrid retrieval incurs additional model-selection costs.

The supplied teacher feedback does not specify a mandatory numerical target. The provenance and timing of any previously proposed 80% target must be confirmed before describing it as a pre-set objective. Future evaluation should use independently labeled questions, checks for critical fee or ratio errors, and an explicit budget.

## Source modules

- `pipeline.py`: the V3.1 embedding, keyword retrieval, candidate fusion, evidence selection, answer generation, and audit workflow.
- `reproduce.py`: reads cached embeddings, recomputes all 50 retrieval rankings, checks them against the saved experiment, and evaluates the recorded results without model calls.
- `run_live.py`: the original command-line entry point. Notebook 02 provides a direct Colab entry point with hidden API-key input.

Notebook 02 extracts these modules automatically. Notebook 06 displays their complete source with file-level explanations. API execution is disabled by default, and the package contains no API key.

## Submission files

| File | Purpose |
|---|---|
| `00_Status_and_Submission_Map_V3.1.ipynb` | Teacher requirements, file mapping, and current status |
| `README_V3.1.ipynb` | Run instructions and product documentation |
| `02_Main_Code_and_API_V3.1.ipynb` | Runnable project and optional API entry point |
| `03_Data_and_Corpus_V3.1.ipynb` | Corpus, document sources, and all 50 questions |
| `04_Evals_and_Gold_V3.1.ipynb` | Original gold labels and evaluation definitions |
| `05_Results_and_Failure_Analysis_V3.1.ipynb` | Results, answer comparisons, failure analysis, and costs |
| `06_Module_Source_and_Explanation_V3.1.ipynb` | Complete module source and explanations |
| `07_Human_Checks_and_Final_Checklist_V3.1.ipynb` | Human-review form and remaining submission tasks |
| `report/Consumer_Lending_V3.1_Report.pdf` and `report/Consumer_Lending_V3.1_Report.md` | Report draft for personal confirmation |
| `video/Demo_Script.md` and `video/RECORDING_GUIDE.md` | Video script and recording guide; actual video still required |


## Source Verification Update — 2026-10-03

See `08_Regulatory_Source_Verification_V3.1.ipynb`. The archive contains nine files for eight instruments, including two D04 copies. D07 is jointly issued by NFRA and the People's Bank of China. The historical corpus date fields are retained for reproduction: D03 (2021-02-19) and D04 (2022-07-12) are signature dates, not uniform effective dates. Source checking does not validate all 199 chunks or all 50 gold labels. The six personal review judgments in notebook 07 are confirmed by LI JIAYI, dated 2026-10-03.


## Verification Update — 2026-10-03

The AI-assisted review of all 50 gold mappings and their key source clauses is completed in `09_Gold_and_Key_Clause_Verification_V3.1.ipynb`. D07's clipped text was supplemented by an indexed government republication. D04_S08 has a substantive new/existing-business transcription error; D05_S06/S08 have abbreviated source references; D02_A70 has footer noise. Corrected source text and recommended labels are stored separately in notebook 09. Notebook 02 intentionally retains the historical experiment; original metrics are unchanged. Full independent human review of all gold labels has not been claimed. The revised report has been approved by the student and is included in this package. The student supplies the demonstration recording separately.
